# Model 1 on real data: CESAW (REAL, professional developers)

Component 1 · Anchor AI (J26-IT-366). Everything here is made from **REAL** data (CESAW, SEI/CMU), never from the synthetic students.

**What this notebook answers.** Model 1 learns how far a person's time estimates are off. Until now we only tested it on synthetic students. CESAW is a public record of about 60,000 real tasks that professional software developers planned and then actually did. If Model 1 predicts their real overruns better than "trust the plan", that is evidence it works outside our own simulation.

**Limits, in plain words.** These are professional developers, not students. CESAW has no task priority and no task type, so every task is treated as one type (`Digital`) with the same priority. The data licence is not stated, so the data is **not** in the repository: you download it yourself.

**Setup (run first).** Clones the C1 branch `dev/ravindu` of the team repo and installs Component 1's pinned requirements. If the repository is private, Colab asks you to sign in to GitHub. The cell prints the Python version: C1 supports 3.11–3.13.

In [ ]:
import sys
print("Python", sys.version.split()[0])
!git clone --depth 1 -b dev/ravindu https://github.com/DulanjaR/AnchorAI.git
%cd AnchorAI
!pip install -q -r components/c1_disruption/requirements.txt

**Get the data.** Download `CESAW_task_fact.csv` from https://kilthub.cmu.edu/articles/CESAW_Project_Data_Fact_sheets/9922697 and put it in a folder outside the repository (in Colab: upload it, or download it with the cell below). Then set `CESAW_PATH` to that folder. Do not commit the file.

In [ ]:
import os
os.makedirs("/content/cesaw", exist_ok=True)
!curl -sL -o /content/cesaw/CESAW_task_fact.csv https://ndownloader.figshare.com/files/17852276
CESAW_PATH = "/content/cesaw"

**Turn tasks into Model 1 records.** One CESAW task becomes one record: the developer is the "student", the planned minutes are the estimate, the actual minutes are the real time (CESAW already leaves out interruptions), and the start time gives the hour and weekday. Tasks with no usable plan, no actual time or no start time are left out, and the counts are shown.

In [ ]:
from components.c1_disruption.ml.drift.external import cesaw
records, dropped = cesaw.load_cesaw(CESAW_PATH)
shape = cesaw.describe(records)
print(dropped)
print(shape)

**How often are real estimates wrong?** The log of (actual ÷ planned) is 0 when the plan was right, positive when the task ran over, negative when it was quicker. A wide spread means estimates are very uncertain, so there is something for Model 1 to learn.

In [ ]:
import numpy as np
from components.c1_disruption.ml.drift.data import log_drift
y = log_drift(records)
print(np.percentile(y, [10, 25, 50, 75, 90]).round(2))

**Run the same test as for the synthetic students.** *Cold start* holds one developer out completely and predicts them from everyone else (developers are held out in 10 groups to keep it fast). *Warm* predicts a developer's later days from their own earlier days plus everyone else. The model is compared with four simple rules: B0 trust the plan, B1 the overall average error, B2 each person's own average, B3 the average plus a person-specific correction without the forest. Lower error is better; the range should contain about 80% of outcomes. This takes a long time (about an hour), because it trains hundreds of models on 60,000 tasks. For a quick look, use `warm_sample=` with a few developer ids and `folds=2`.

In [ ]:
report = cesaw.evaluate(records)

**Results (REAL, CESAW, professional developers).** The table is also saved as `docs/results/c1_model1_cesaw.md` in the repository.

In [ ]:
text = cesaw.to_markdown(report, dropped, shape)
print(text)